
# EXPERIMENT 3 – EDA: Data Cleaning and Data Transformation

**Dataset:** IPL Ball-by-Ball Dataset  
**Tools:** Python 3.x, Jupyter Notebook, Pandas, NumPy, Scikit-Learn, Matplotlib

## Aim
To identify and handle missing values, duplicate records, outliers, inconsistent data, and perform data transformation techniques such as normalization and encoding using Python Pandas.

## IPL Adaptation
The lab-manual employee dataset is replaced with the IPL `matches.csv` and `deliveries.csv` files. The same data-cleaning and transformation concepts are demonstrated on IPL match data.



## Theory

Real-world datasets often contain:
- Missing Values
- Duplicate Records
- Incorrect Data Types
- Outliers
- Inconsistent Formatting

Data cleaning improves data quality and increases the accuracy of analytical models. Data transformation converts data into suitable formats for analysis and machine learning.

### Learning Outcomes
- Detect missing values
- Handle null values
- Remove duplicates
- Identify outliers
- Normalize data
- Encode categorical variables
- Prepare datasets for analytics


## 1. Import Libraries and Load IPL Data

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, LabelEncoder

matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)


## 2. Inspect the Dataset

In [ ]:

print("Matches columns:")
print(matches.columns.tolist())

print("\nDeliveries columns:")
print(deliveries.columns.tolist())

display(matches.head())
display(deliveries.head())


## Algorithm 1 – Identifying Missing Values

1. Import Pandas.
2. Load the dataset.
3. Use `isnull()`.
4. Count missing values.
5. Display the result.

In [ ]:

print("Missing values in matches.csv:")
display(matches.isnull())

print("Missing-value count:")
display(matches.isnull().sum())

print("\nMissing values in deliveries.csv:")
display(deliveries.isnull().sum())


## 3. Handle Missing Values

For numeric columns, missing values can be replaced using the mean. For categorical columns, the mode can be used. The code below applies this safely only to columns that actually exist in the IPL file.

In [ ]:

matches_clean = matches.copy()

numeric_cols = matches_clean.select_dtypes(include=np.number).columns
categorical_cols = matches_clean.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols:
    if matches_clean[col].isnull().any():
        matches_clean[col] = matches_clean[col].fillna(matches_clean[col].mean())

for col in categorical_cols:
    if matches_clean[col].isnull().any():
        mode_value = matches_clean[col].mode()
        if not mode_value.empty:
            matches_clean[col] = matches_clean[col].fillna(mode_value.iloc[0])

print("Remaining missing values:")
display(matches_clean.isnull().sum())


## Algorithm 2 – Replacing Missing Values

1. Calculate a mean value for numerical data.
2. Replace null values.
3. Verify the replacement.

In [ ]:

before_missing = matches.isnull().sum()
after_missing = matches_clean.isnull().sum()

comparison = pd.DataFrame({
    "Before Cleaning": before_missing,
    "After Cleaning": after_missing
})

display(comparison)


## Algorithm 3 – Removing Duplicate Records

1. Use `duplicated()`.
2. Count duplicate records.
3. Remove duplicates using `drop_duplicates()`.
4. Verify the dataset.

In [ ]:

duplicate_count = matches_clean.duplicated().sum()
print("Duplicate rows before removal:", duplicate_count)

matches_clean = matches_clean.drop_duplicates().copy()

print("Duplicate rows after removal:", matches_clean.duplicated().sum())
print("Cleaned matches shape:", matches_clean.shape)


## 4. Detecting Outliers

The lab manual uses the Inter Quartile Range (IQR) method.

**IQR = Q3 − Q1**

**Lower Limit = Q1 − 1.5 × IQR**

**Upper Limit = Q3 + 1.5 × IQR**

For IPL data, a suitable numerical column is selected automatically.

In [ ]:

# Prefer common IPL numerical columns when available.
preferred_numeric = ["total_runs", "total_runs_y", "batsman_runs", "extra_runs"]

available = [c for c in preferred_numeric if c in deliveries.columns]

if available:
    outlier_col = available[0]
else:
    numeric_delivery_cols = deliveries.select_dtypes(include=np.number).columns.tolist()
    outlier_col = numeric_delivery_cols[0] if numeric_delivery_cols else None

print("Column selected for IQR outlier detection:", outlier_col)

if outlier_col:
    q1 = deliveries[outlier_col].quantile(0.25)
    q3 = deliveries[outlier_col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = deliveries[
        (deliveries[outlier_col] < lower) |
        (deliveries[outlier_col] > upper)
    ]

    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower limit:", lower)
    print("Upper limit:", upper)
    print("Number of outliers:", len(outliers))
    display(outliers.head(10))
else:
    print("No numerical column is available for outlier detection.")


## Visualization of Outliers

In [ ]:

if outlier_col:
    plt.figure(figsize=(8, 4))
    plt.boxplot(deliveries[outlier_col].dropna())
    plt.title(f"IPL {outlier_col} – Outlier Detection")
    plt.ylabel(outlier_col)
    plt.show()


## Algorithm 5 – Normalization

**Purpose:** Convert numerical values into a common scale.

Min-Max normalization:

**X_norm = (X − X_min) / (X_max − X_min)**

The manual uses `MinMaxScaler` from Scikit-Learn.

In [ ]:

# Select a useful numerical IPL column for normalization.
normalization_candidates = [
    c for c in ["total_runs", "total_runs_y", "batsman_runs", "extra_runs"]
    if c in deliveries.columns
]

if normalization_candidates:
    norm_col = normalization_candidates[0]

    scaler = MinMaxScaler()
    deliveries_transformed = deliveries.copy()
    deliveries_transformed[norm_col + "_normalized"] = scaler.fit_transform(
        deliveries_transformed[[norm_col]]
    )

    display(deliveries_transformed[[norm_col, norm_col + "_normalized"]].head(10))
else:
    print("No suitable numerical IPL column found for normalization.")


## Algorithm 6 – Label Encoding

**Purpose:** Convert categorical data into numerical format.

In [ ]:

# Use a categorical IPL column if available.
label_candidates = [
    c for c in ["winner", "team1", "team2", "city", "venue", "toss_winner"]
    if c in matches_clean.columns
]

if label_candidates:
    label_col = label_candidates[0]
    encoder = LabelEncoder()

    encoded_values = encoder.fit_transform(
        matches_clean[label_col].astype(str)
    )

    label_encoding_result = pd.DataFrame({
        label_col: matches_clean[label_col].astype(str),
        label_col + "_encoded": encoded_values
    })

    print("Column encoded:", label_col)
    display(label_encoding_result.head(10))
else:
    print("No suitable categorical IPL column found for label encoding.")


## Program – One-Hot Encoding

One-hot encoding represents categories as binary columns using `pd.get_dummies()`.

In [ ]:

if label_candidates:
    one_hot = pd.get_dummies(matches_clean[label_col].astype(str))
    display(one_hot.head(10))
else:
    print("No suitable categorical IPL column found for one-hot encoding.")


## 5. Cleaning and Transformation Summary

In [ ]:

summary = {
    "Original matches rows": len(matches),
    "Cleaned matches rows": len(matches_clean),
    "Duplicate rows removed": duplicate_count,
    "Missing values before": int(matches.isnull().sum().sum()),
    "Missing values after": int(matches_clean.isnull().sum().sum()),
}

if outlier_col:
    summary["Outlier detection column"] = outlier_col
    summary["Outliers detected"] = int(len(outliers))

display(pd.DataFrame([summary]))



## Data Transformation Summary

| Technique | Purpose |
|---|---|
| Missing-value handling | Improve data completeness |
| Duplicate removal | Remove repeated records |
| Outlier detection | Improve data quality |
| Normalization | Scale numerical data |
| Label Encoding | Convert categories to numeric labels |
| One-Hot Encoding | Represent categories as binary columns |



## Applications

- **Healthcare:** Cleaning patient records
- **Banking:** Removing duplicate transactions
- **Education:** Student performance analysis
- **Retail:** Sales data preprocessing
- **Social Media:** User behavior analysis

## Advantages

1. Improves Data Quality
2. Increases Model Accuracy
3. Reduces Errors
4. Removes Noise
5. Enables Better Analytics



## Viva Questions

1. **What is Data Cleaning?**  
   The process of correcting inaccurate or incomplete data.

2. **What are Missing Values?**  
   Values that are absent from the dataset.

3. **What is the purpose of `fillna()`?**  
   To replace missing values.

4. **What is a duplicate record?**  
   A repeated row in a dataset.

5. **Which function removes duplicates?**  
   `drop_duplicates()`

6. **What is an outlier?**  
   An observation significantly different from others.

7. **What is normalization?**  
   Scaling numerical values to a standard range.

8. **What is Min-Max Scaling?**  
   A normalization technique that converts values between 0 and 1.

9. **What is Label Encoding?**  
   Converting categorical values into numeric labels.

10. **What is One-Hot Encoding?**  
    Representing categories as binary columns.

11. **Why is data cleaning important?**  
    To improve the quality and reliability of data analysis.

12. **Which library provides `MinMaxScaler`?**  
    Scikit-Learn.



## Result

Thus, missing values, duplicate records, and outliers were identified and handled successfully. Data transformation techniques such as normalization and encoding were performed on the IPL dataset, preparing the data for further exploratory data analysis and machine learning applications.
